In [1]:
import pandas as pd 

In [6]:
def create_clean_df_for_Content_pipeline(movies_df,tags_df):
    new_movies_df = movies_df[['movieId','title','genres']].copy()
    # cleaning movies_df
    new_movies_df['title'] = (
        movies_df['title'].fillna("")
        .astype(str)
        .str.strip()
    )
    new_movies_df['genres'] = (
        new_movies_df['genres'].fillna("")
        .astype(str)
        .str.replace("|"," ")
        .str.strip()
    )
    
    # cleaning tags_df
    new_tags_df = tags_df[["movieId","tag"]].copy()
    new_tags_df.dropna(subset=['movieId','tag'])
    
    new_tags_df['tag'] = (
    new_tags_df['tag']
    .astype(str)
    .str.strip()
    .str.lower()
    )
    
    new_tags_df = new_tags_df[new_tags_df['tag'] != ""]
    
    tags_grouped = (
    new_tags_df.groupby('movieId')['tag']
    .apply(lambda crr_mov_tags : " ".join(crr_mov_tags)) # this lambda function input tags of first movie and join them with " " and then moves to next movie tags and do same upto so on
    .reset_index(name='tags') # it cut out the movieId from the resultant series into separate column and the name="tags" changes the name of remaining tag  series to tagss
)
    
    # merging the tags and movies_df
    movies_df_final = pd.merge(
        new_movies_df,
        tags_grouped,
        on='movieId',
        how='left',
        validate='one_to_one' # validating so that each movieId appears one in both data frames otherwise it will raise error 
    )
    
    # cleaning final frame 
    movies_df_final["tags"] = movies_df_final['tags'].fillna("")
    movies_df_final = movies_df_final.drop_duplicates(subset=["movieId"])
    movies_df_final = movies_df_final.reset_index(drop=True)
    
    return movies_df_final


In [7]:
movies = pd.read_csv(r'../datasets/movies.csv')
tags = pd.read_csv(r"../datasets/tags.csv")

In [8]:
new_df = create_clean_df_for_Content_pipeline(movies,tags)

In [9]:
new_df

,movieId,title,genres,tags
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game robin williams game
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy Drama Romance,
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake
...,...,...,...,...
9717,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,
9718,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,
9719,193585,Flint (2017),Drama,
9720,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,


In [10]:
new_df['content'] = (
    new_df['title'] + " "
    + new_df['genres'] + " "
    + new_df['tags'] 
) 

In [11]:
new_df

,movieId,title,genres,tags,content
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game robin williams game,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old,Grumpier Old Men (1995) Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy Drama Romance,,Waiting to Exhale (1995) Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Father of the Bride Part II (1995) Comedy preg...
...,...,...,...,...,...
9717,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,,Black Butler: Book of the Atlantic (2017) Acti...
9718,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,,No Game No Life: Zero (2017) Animation Comedy ...
9719,193585,Flint (2017),Drama,,Flint (2017) Drama
9720,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,,Bungo Stray Dogs: Dead Apple (2018) Action Ani...


In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [ ]:

def fetch_movie_metadata(
    links_path="../datasets/links.csv",
    output_path="../datasets/tmdb_movies.csv",
):
    if not TMDB_API_KEY:
        raise ValueError("TMDB_API_KEY is missing from .env")

    links_df = pd.read_csv(links_path)

    # Keep movies that have a TMDB ID
    links_df = links_df.dropna(subset=["tmdbId"]).copy()
    links_df["tmdbId"] = links_df["tmdbId"].astype(int)

    # Reuse previously fetched records when rerunning
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        existing_ids = set(existing_df["tmdbId"].dropna().astype(int))
        records = existing_df.to_dict("records")
    else:
        existing_ids = set()
        records = []

    session = requests.Session()
    session.headers.update({
        "Authorization": f"Bearer {TMDB_API_KEY}",
        "accept": "application/json",
    })
    
    count = 0 
    for row in links_df.itertuples(index=False):
        movie_id = row.movieId
        tmdb_id = row.tmdbId

        if tmdb_id in existing_ids:
            continue

        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

        try:
            response = session.get(
                url,
                params={
                "api_key": TMDB_API_KEY,
                "language": "en-US",
            },
            timeout=15,
            )

            if response.status_code == 404:
                print(f"Movie not found: TMDB ID {tmdb_id}")
                continue

            response.raise_for_status()
            data = response.json()

            records.append({
                "movieId": movie_id,
                "tmdbId": tmdb_id,
                "overview": data.get("overview") or "",
                "release_date": data.get("release_date") or "",
                "poster_path": data.get("poster_path"),
                "backdrop_path": data.get("backdrop_path"),
            })
            count+=1
            print(count)

            existing_ids.add(tmdb_id)

        except requests.RequestException as error:
            print(f"Request failed for TMDB ID {tmdb_id}: {error}")

        # Save progress so successful requests aren't lost
        pd.DataFrame(records).to_csv(output_path, index=False)
        time.sleep(0.0)

    metadata_df = pd.DataFrame(records)
    metadata_df.to_csv(output_path, index=False)

    return metadata_df
    

In [ ]:
fetch_movie_metadata()

Movie not found: TMDB ID 12773
Movie not found: TMDB ID 17882
Movie not found: TMDB ID 68149
Movie not found: TMDB ID 24549
Movie not found: TMDB ID 14980
Movie not found: TMDB ID 164721
Movie not found: TMDB ID 140207
